# Document store MCP tools

Exercises `search_code`, `describe_code`, `search_research_library` and
`describe_research_library` over the wire, the way a client reaches them.

**The server must be running** — in VSCode: *yada MCP server (http)*, or:

```
./start_mcp_server.sh
```

It serves `http://127.0.0.1:8081/mcp`. Override with `YADA_MCP_URL`.

These tools return **no text blocks** — everything is in `structured_content`,
derived from the pydantic return annotation. `show_structured` prints that payload,
which is what a client actually reasons over.

In [1]:
# Pick up edits to utils.py without restarting the kernel. Without this the
# kernel holds the module it first imported, and a change there is invisible
# however many times the SERVER is restarted -- the stale half is the client.
%load_ext autoreload
%autoreload 2

# Says plainly which utils.py this kernel has. `_caller` is what tags a request
# with the helper that sent it; if it is missing, the kernel predates that change
# and the server will log a bare `rpc tools/list <-` with no caller, no matter
# what the server is running. Restart the kernel -- autoreload cannot load
# itself into a kernel that went stale before it was enabled.
import utils

print(utils.__file__)
print('tags requests with the caller:', hasattr(utils, '_caller'))

/Users/troy/Develop/gly.fish/yada/notebooks/mcp/utils.py
tags requests with the caller: True


In [2]:
from utils import (MCP_URL, call_tool, list_mcp_tools, show_structured,
                   show_tool_schema)

MCP_URL

'http://127.0.0.1:8081/mcp'

In [3]:
await list_mcp_tools()

plot: Create a chart from data supplied in the request and return it as an image.
plot_report: Render a stored time series report as a chart and return it as an image.
search_code: Search Troy Stribling's indexed code repositories by meaning and return ranked excerpts with their file paths (10,763 files, 415,006 indexed chunks).
- query: what to look for, in natural language.
- repo: a repository name, or a GitHub account name to search everything under it.
describe_code: What the code repositories contain: accounts, repositories, file types and counts, under any filters given.
search_research_library: Search Troy Stribling's research library -- papers, research notes and blog posts -- by meaning, returning ranked excerpts (64 files, 4,875 indexed chunks).
- query: what to look for, in natural language.
- authors: who wrote it.
describe_research_library: What the research library contains: authors, titles, shelves and counts, under any filters given.


## 1. The published descriptions

What the client sees before it calls anything, and the only thing steering which
tool it picks and what it puts in the arguments. `shelf`'s values and the author
surnames are **generated from the vocabulary at registration** — a hand-written list
goes stale the first time a document is added, and goes stale silently.

In [4]:
await show_tool_schema('search_code')

search_code
  Search Troy Stribling's indexed code repositories by meaning and return ranked excerpts with their file paths (10,763 files, 415,006 indexed chunks).
- query: what to look for, in natural language.
- repo: a repository name, or a GitHub account name to search everything under it. A partial name works. Omit to search every repository. Call describe_code to see what exists.
- ext: a file extension such as '.py', or a language name such as 'Python', which maps to the extensions that language uses. Pass '(no extension)' for files that have none, such as LICENSE and Makefile. Omit to search all types.
A value that matches nothing is reported with what does exist, and no search is run.

  query                string   required
  repo                 any      optional
  ext                  any      optional
  limit                integer  optional, default=10


{'properties': {'query': {'title': 'Query', 'type': 'string'},
  'repo': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Repo'},
  'ext': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Ext'},
  'limit': {'default': 10, 'title': 'Limit', 'type': 'integer'}},
 'required': ['query'],
 'type': 'object',
 'title': 'search_codeArguments'}

In [ ]:
await show_tool_schema('describe_code')

In [ ]:
await show_tool_schema('search_research_library')

In [ ]:
await show_tool_schema('describe_research_library')

## 2. What the code store holds

Unfiltered, this is the whole picture: two accounts, 101 repositories, 190 extensions.

Watch the **`account/repo`** facet. A flat facet can say there are two accounts and
101 repositories but not *which holds which*, and a model asked for a pairing it
cannot see will invent one. Asked for a table of account, project and size, Goose
returned correct repository names and chunk counts with an invented account column —
15 of 20 rows wrong, each fork attributed to its upstream owner (`quantopian` for
pyalgotrade, `mementum` for backtrader) rather than the account that holds this copy.

In [ ]:
show_structured(await call_tool('describe_code'), facets=15)

## 3. Filtered describes

`describe_code(repo='zgomot')` answers *what is this repository written in*. The
answer comes from the per-combination rows, which is why it can be filtered at all:
per-field marginals cannot answer a conjunction, and asking Chroma the same question
costs 3.8 s for a filter matching 384 chunks.

In [ ]:
show_structured(await call_tool('describe_code', {'repo': 'zgomot'}))

### Two resolutions in one call

`repo='glyfish'` is an **account**, not a repository — the server classifies the name
rather than making the caller do it, and applies `{'account': 'glyfish'}`, one equality
instead of a 22-value `$in`. `ext='Python'` is a **language**, mapped to the extensions
that language uses. Both decisions are reported in `filters_resolved`.

In [ ]:
show_structured(await call_tool(
    'describe_code', {'repo': 'glyfish', 'ext': 'Python'}), facets=10)

### A value that matches nothing

Answered from the vocabulary in microseconds, with what *does* exist. No search runs.
This is the case that used to cost 3 s and return a confident wrong answer: Chroma's
`where` is exact equality, so an unresolvable value matches zero rows and raises
nothing, which is indistinguishable from an empty corpus.

In [ ]:
show_structured(await call_tool('describe_code', {'repo': 'nonesuch'}))

### Partial names and the extension with no name

`zgom` resolves by substring. A blank `ext` means *files that have none* — LICENSE,
Makefile, Rakefile — which is a real category and the opposite of omitting `ext`,
which searches every extension.

In [ ]:
show_structured(await call_tool('describe_code', {'repo': 'zgom'}))

In [ ]:
show_structured(await call_tool('describe_code', {'ext': '(no extension)'}), facets=8)

## 4. Searching code

Ranked excerpts with their metadata and scores. `retrieved` is how many candidates
were considered, `best_score` the top relevance before filtering — and `facets` here
describes **the hits that came back**, not the store.

In [ ]:
show_structured(await call_tool('search_code', {
    'query': 'How is the MIDI clock implemented?',
    'repo': 'zgomot',
    'limit': 5,
}), hits=3)

## 5. The research library

`authors` is where this work started. The field stores the **whole author list** as
one value — `'Troy Stribling, William H. Matthaeus'` — so naming one contributor
matched nothing and returned no error. A surname now resolves to every spelling and
every co-authored paper.

In [ ]:
show_structured(await call_tool('describe_research_library'), facets=8)

In [ ]:
show_structured(await call_tool('search_research_library', {
    'query': 'turbulence anisotropy',
    'authors': 'Matthaeus',
    'limit': 5,
}), hits=3)

### Scope the negative to the claim

`authors='Einstein'` is answered without searching. The honest statement is *no
document has Einstein as an author* — not *nothing about Einstein*, which is false:
a content search for Einstein returns Brownian motion documents. `available` is the
next step rather than a dead end.

In [ ]:
show_structured(await call_tool('search_research_library', {
    'query': 'relativity', 'authors': 'Einstein'}))

In [ ]:
show_structured(await call_tool('describe_research_library', {'shelf': 'jaynes'}), facets=8)

## 6. The raw payload

What a client parses, unrendered — `output_schema` is derived from the return
annotation and `structured_content` is populated by the SDK, so this is the contract.

In [ ]:
import json

result = await call_tool('describe_code', {'repo': 'zgomot', 'ext': '.rb'})
print(json.dumps(result.structured_content, indent=2)[:1800])